In [4]:
import matplotlib.pyplot as plt
import numpy as np
import os

stats = {
    "original_simple": "..\\stats\\original_simple.txt",
    "original_deep": "..\\stats\\original_deep.txt",
    "balanced": "..\\stats\\balanced.txt",
    "augmented": "..\\stats\\augmented.txt",
    "combined": "..\\stats\\combined.txt",
    "final": "..\\stats\\final.txt"
}

class_mapping = {
    "0": "0", "1": "1", "2": "2", "3": "3", "4": "4", "5": "5", 
    "6": "6", "7": "7", "8": "8", "9": "9", "10": "A", "11": "B",
    "12": "C", "13": "D", "14": "E", "15": "F", "16": "G", "17": "H",
    "18": "I", "19": "J", "20": "K", "21": "L", "22": "M", "23": "N",
    "24": "O", "25": "P", "26": "Q", "27": "R", "28": "S", "29": "T",
    "30": "U", "31": "V", "32": "W", "33": "X", "34": "Y", "35": "Z",
    "36": "a", "37": "b", "38": "c", "39": "d", "40": "e", "41": "f",
    "42": "g", "43": "h", "44": "i", "45": "j", "46": "k", "47": "l",
    "48": "m", "49": "n", "50": "o", "51": "p", "52": "q", "53": "r",
    "54": "s", "55": "t", "56": "u", "57": "v", "58": "w", "59": "x",
    "60": "y", "61": "z"
}

def parse_classification_report(file_path):
    """Đọc và parse file classification report"""
    classes = []
    precisions = []
    recalls = []
    f1_scores = []
    supports = []
    
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line or 'precision' in line or 'accuracy' in line or 'macro avg' in line or 'weighted avg' in line:
                    continue

                parts = line.split()
                if len(parts) >= 5:
                    try:
                        class_idx = parts[0]
                        if class_idx.isdigit() and int(class_idx) <= 61:
                            classes.append(class_mapping.get(class_idx, class_idx))
                            precisions.append(float(parts[1]))
                            recalls.append(float(parts[2]))
                            f1_scores.append(float(parts[3]))
                            supports.append(int(parts[4]))
                    except (ValueError, IndexError):
                        continue
    except FileNotFoundError:
        print(f"File not found: {file_path}")
        return [], [], [], [], []
    
    return classes, precisions, recalls, f1_scores, supports

def compare_all_models(stats_dict, output_dir='visualizations'):    
    os.makedirs(output_dir, exist_ok=True)
    
    all_models_data = {}
    
    for model_name, file_path in stats_dict.items():
        classes, precisions, recalls, f1_scores, supports = parse_classification_report(file_path)
        if classes:
            all_models_data[model_name] = {
                'classes': classes,
                'precisions': precisions,
                'recalls': recalls,
                'f1_scores': f1_scores,
                'supports': supports,
                'avg_precision': np.mean(precisions),
                'avg_recall': np.mean(recalls),
                'avg_f1': np.mean(f1_scores)
            }
    
    plot_model_comparison_summary(all_models_data, output_dir)
    
    for model_name, data in all_models_data.items():
        plot_single_model_detailed(model_name, data, output_dir)
    
    plot_f1_comparison_by_class(all_models_data, output_dir)
    
    print_summary_report(all_models_data)


def plot_model_comparison_summary(all_models_data, output_dir):
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Model Performance Comparison - Summary', fontsize=16, fontweight='bold')
    
    model_names = list(all_models_data.keys())
    avg_precisions = [data['avg_precision'] for data in all_models_data.values()]
    avg_recalls = [data['avg_recall'] for data in all_models_data.values()]
    avg_f1s = [data['avg_f1'] for data in all_models_data.values()]
    
    ax1 = axes[0, 0]
    x = np.arange(len(model_names))
    width = 0.25
    
    ax1.bar(x - width, avg_precisions, width, label='Precision', alpha=0.8, color='#3498db')
    ax1.bar(x, avg_recalls, width, label='Recall', alpha=0.8, color='#2ecc71')
    ax1.bar(x + width, avg_f1s, width, label='F1-Score', alpha=0.8, color='#e74c3c')
    
    ax1.set_xlabel('Models', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Average Score', fontsize=12, fontweight='bold')
    ax1.set_title('Average Metrics Comparison', fontsize=13, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(model_names, rotation=45, ha='right')
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)
    ax1.set_ylim([0, 1])
    
    ax2 = axes[0, 1]
    colors = plt.cm.Set3(np.linspace(0, 1, len(model_names)))
    bars = ax2.barh(model_names, avg_f1s, color=colors, alpha=0.8, edgecolor='black')
    ax2.set_xlabel('F1-Score', fontsize=12, fontweight='bold')
    ax2.set_title('F1-Score Ranking', fontsize=13, fontweight='bold')
    ax2.set_xlim([0, 1])
    ax2.grid(axis='x', alpha=0.3)
    
    for i, (bar, val) in enumerate(zip(bars, avg_f1s)):
        ax2.text(val + 0.01, bar.get_y() + bar.get_height()/2, 
                f'{val:.4f}', va='center', fontweight='bold')
    
    ax3 = axes[1, 0]
    for i, (name, data) in enumerate(all_models_data.items()):
        ax3.scatter(data['avg_recall'], data['avg_precision'], 
                   s=300, alpha=0.6, label=name, color=colors[i], edgecolors='black', linewidth=2)
    
    ax3.set_xlabel('Average Recall', fontsize=12, fontweight='bold')
    ax3.set_ylabel('Average Precision', fontsize=12, fontweight='bold')
    ax3.set_title('Precision vs Recall Trade-off', fontsize=13, fontweight='bold')
    ax3.legend(loc='best', fontsize=9)
    ax3.grid(True, alpha=0.3)
    ax3.set_xlim([0.7, 0.85])
    ax3.set_ylim([0.7, 0.85])
    
    lims = [0.7, 0.85]
    ax3.plot(lims, lims, 'k--', alpha=0.3, linewidth=1)
    
    ax4 = axes[1, 1]
    poor_classes_count = []
    for name, data in all_models_data.items():
        count = sum(1 for f1 in data['f1_scores'] if f1 < 0.5)
        poor_classes_count.append(count)
    
    bars = ax4.bar(model_names, poor_classes_count, color=colors, alpha=0.8, edgecolor='black')
    ax4.set_ylabel('Number of Classes', fontsize=12, fontweight='bold')
    ax4.set_title('Classes with F1-Score < 0.5', fontsize=13, fontweight='bold')
    ax4.set_xticklabels(model_names, rotation=45, ha='right')
    ax4.grid(axis='y', alpha=0.3)
    
    for bar, val in zip(bars, poor_classes_count):
        if val > 0:
            ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                    str(val), ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    output_path = os.path.join(output_dir, 'model_comparison_summary.png')
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()


def plot_single_model_detailed(model_name, data, output_dir):
    fig = plt.figure(figsize=(20, 12))
    fig.suptitle(f'Detailed Analysis - {model_name.upper()}', fontsize=16, fontweight='bold')
    
    classes = data['classes']
    precisions = data['precisions']
    recalls = data['recalls']
    f1_scores = data['f1_scores']
    supports = data['supports']
    
    # 1. Bar chart Precision, Recall, F1-Score
    ax1 = plt.subplot(2, 2, 1)
    x = np.arange(len(classes))
    width = 0.25
    
    ax1.bar(x - width, precisions, width, label='Precision', alpha=0.8, color='#3498db')
    ax1.bar(x, recalls, width, label='Recall', alpha=0.8, color='#2ecc71')
    ax1.bar(x + width, f1_scores, width, label='F1-Score', alpha=0.8, color='#e74c3c')
    
    ax1.set_xlabel('Classes', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax1.set_title('Metrics by Class', fontsize=13, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(classes, fontsize=8)
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)
    ax1.set_ylim([0, 1.05])
    
    # 2. F1-Score distribution
    ax2 = plt.subplot(2, 2, 2)
    colors = ['red' if f1 < 0.5 else 'orange' if f1 < 0.7 else 'green' for f1 in f1_scores]
    ax2.scatter(range(len(f1_scores)), f1_scores, c=colors, s=100, alpha=0.6, edgecolors='black')
    ax2.plot(range(len(f1_scores)), f1_scores, 'b-', alpha=0.3, linewidth=1)
    ax2.axhline(y=0.7, color='orange', linestyle='--', linewidth=2, label='Good (0.7)')
    ax2.axhline(y=0.5, color='red', linestyle='--', linewidth=2, label='Poor (0.5)')
    ax2.set_xlabel('Class Index', fontsize=12, fontweight='bold')
    ax2.set_ylabel('F1-Score', fontsize=12, fontweight='bold')
    ax2.set_title('F1-Score Distribution', fontsize=13, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. Support distribution
    ax3 = plt.subplot(2, 2, 3)
    colors_support = ['#e74c3c' if s < 500 else '#f39c12' if s < 2000 else '#2ecc71' for s in supports]
    ax3.bar(classes, supports, color=colors_support, alpha=0.7, edgecolor='black')
    ax3.set_xlabel('Classes', fontsize=12, fontweight='bold')
    ax3.set_ylabel('Support', fontsize=12, fontweight='bold')
    ax3.set_title('Class Distribution', fontsize=13, fontweight='bold')
    ax3.set_xticklabels(classes, fontsize=8)
    ax3.grid(axis='y', alpha=0.3)
    
    # 4. Support vs F1-Score
    ax4 = plt.subplot(2, 2, 4)
    scatter = ax4.scatter(supports, f1_scores, c=f1_scores, cmap='RdYlGn', 
                         s=100, alpha=0.6, edgecolors='black')
    ax4.set_xlabel('Support', fontsize=12, fontweight='bold')
    ax4.set_ylabel('F1-Score', fontsize=12, fontweight='bold')
    ax4.set_title('Support vs F1-Score', fontsize=13, fontweight='bold')
    ax4.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=ax4, label='F1-Score')
    
    # Annotate worst 5 classes
    worst_indices = np.argsort(f1_scores)[:5]
    for idx in worst_indices:
        ax4.annotate(classes[idx], (supports[idx], f1_scores[idx]),
                    xytext=(5, 5), textcoords='offset points', fontsize=9,
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.5))
    
    plt.tight_layout()
    output_path = os.path.join(output_dir, f'{model_name}_detailed.png')
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()


def plot_f1_comparison_by_class(all_models_data, output_dir):
    fig, axes = plt.subplots(2, 1, figsize=(24, 14))
    fig.suptitle('F1-Score Comparison Across All Models by Class', fontsize=16, fontweight='bold')
    
    first_model = list(all_models_data.values())[0]
    classes = first_model['classes']
    x = np.arange(len(classes))
    
    model_names = list(all_models_data.keys())
    colors = plt.cm.Set3(np.linspace(0, 1, len(model_names)))
    
    ax1 = axes[0]
    for i, (name, data) in enumerate(all_models_data.items()):
        ax1.plot(x, data['f1_scores'], marker='o', linewidth=2, 
                label=name, color=colors[i], alpha=0.7, markersize=4)
    
    ax1.axhline(y=0.7, color='orange', linestyle='--', linewidth=1.5, alpha=0.5, label='Good threshold')
    ax1.axhline(y=0.5, color='red', linestyle='--', linewidth=1.5, alpha=0.5, label='Poor threshold')
    ax1.set_xlabel('Classes', fontsize=12, fontweight='bold')
    ax1.set_ylabel('F1-Score', fontsize=12, fontweight='bold')
    ax1.set_title('F1-Score Comparison - Line Plot', fontsize=13, fontweight='bold')
    ax1.set_xticks(x[::5])
    ax1.set_xticklabels([classes[i] for i in range(0, len(classes), 5)])
    ax1.legend(loc='best', ncol=3, fontsize=10)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim([0, 1.05])
    
    ax2 = axes[1]
    
    all_f1s = []
    for data in all_models_data.values():
        all_f1s.extend(data['f1_scores'])
    
    problem_classes_idx = set()
    for data in all_models_data.values():
        for i, f1 in enumerate(data['f1_scores']):
            if f1 < 0.6:
                problem_classes_idx.add(i)
    
    problem_classes_idx = sorted(list(problem_classes_idx))[:20]  
    problem_classes = [classes[i] for i in problem_classes_idx]
    
    x_bar = np.arange(len(problem_classes))
    width = 0.8 / len(model_names)
    
    for i, (name, data) in enumerate(all_models_data.items()):
        problem_f1s = [data['f1_scores'][idx] for idx in problem_classes_idx]
        offset = width * (i - len(model_names)/2 + 0.5)
        ax2.bar(x_bar + offset, problem_f1s, width, label=name, 
               color=colors[i], alpha=0.8, edgecolor='black', linewidth=0.5)
    
    ax2.set_xlabel('Problem Classes (F1 < 0.6)', fontsize=12, fontweight='bold')
    ax2.set_ylabel('F1-Score', fontsize=12, fontweight='bold')
    ax2.set_title('Detailed Comparison - Challenging Classes', fontsize=13, fontweight='bold')
    ax2.set_xticks(x_bar)
    ax2.set_xticklabels(problem_classes)
    ax2.legend(loc='best', ncol=3, fontsize=10)
    ax2.grid(axis='y', alpha=0.3)
    ax2.set_ylim([0, 1.05])
    ax2.axhline(y=0.5, color='red', linestyle='--', alpha=0.3)
    
    plt.tight_layout()
    output_path = os.path.join(output_dir, 'f1_comparison_by_class.png')
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()


def print_summary_report(all_models_data):
    print("\n" + "="*80)
    print("SUMMARY REPORT - ALL MODELS")
    print("="*80)

    sorted_models = sorted(all_models_data.items(), 
                          key=lambda x: x[1]['avg_f1'], reverse=True)
    
    print(f"\n{'Model':<20} {'Avg Precision':>15} {'Avg Recall':>15} {'Avg F1-Score':>15} {'Classes F1<0.5':>18}")
    print("-"*80)
    
    for name, data in sorted_models:
        poor_count = sum(1 for f1 in data['f1_scores'] if f1 < 0.5)
        print(f"{name:<20} {data['avg_precision']:>15.4f} {data['avg_recall']:>15.4f} "
              f"{data['avg_f1']:>15.4f} {poor_count:>18}")
    
    print("\n" + "="*80)
    print("BEST MODEL BY EACH METRIC:")
    print("-"*80)
    
    best_precision = max(sorted_models, key=lambda x: x[1]['avg_precision'])
    best_recall = max(sorted_models, key=lambda x: x[1]['avg_recall'])
    best_f1 = max(sorted_models, key=lambda x: x[1]['avg_f1'])
    
    print(f"Best Precision: {best_precision[0]} ({best_precision[1]['avg_precision']:.4f})")
    print(f"Best Recall:    {best_recall[0]} ({best_recall[1]['avg_recall']:.4f})")
    print(f"Best F1-Score:  {best_f1[0]} ({best_f1[1]['avg_f1']:.4f})")
    
    print("="*80 + "\n")
compare_all_models(stats, output_dir='visualizations')

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_13672\3884390342.py:153: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax4.set_xticklabels(model_names, rotation=45, ha='right')
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_13672\3884390342.py:215: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax3.set_xticklabels(classes, fontsize=8)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_13672\3884390342.py:215: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax3.set_xticklabels(classes, fontsize=8)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_13672\3884390342.py:215: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax3.set_xticklabels(classes, fontsize=8)
C:\Users\ADMIN\AppData\


SUMMARY REPORT - ALL MODELS

Model                  Avg Precision      Avg Recall    Avg F1-Score     Classes F1<0.5
--------------------------------------------------------------------------------
original_simple               0.8286          0.7674          0.7697                  7
combined                      0.7517          0.7603          0.7481                 10
final                         0.7521          0.7589          0.7472                 10
original_deep                 0.8092          0.7298          0.7321                 11
augmented                     0.7956          0.7286          0.7303                 11
balanced                      0.7304          0.7778          0.7280                 11

BEST MODEL BY EACH METRIC:
--------------------------------------------------------------------------------
Best Precision: original_simple (0.8286)
Best Recall:    balanced (0.7778)
Best F1-Score:  original_simple (0.7697)

